# Aligner deux séquences : l'optimum se calcule, il ne se devine pas

Comparer deux séquences d'ADN ou de protéines, c'est chercher un **appariement de caractères** qui maximise un score : accords récompensés, désaccords et brèches pénalisés. L'alignement optimal n'est pas celui qui « ressemble » le plus à l'œil : c'est celui dont le score est le plus grand parmi tous les alignements possibles, et ce maximum se calcule exactement par programmation dynamique.

Ce carnet écrit deux algorithmes classiques à partir de zéro — Needleman-Wunsch (alignement global) et Smith-Waterman (alignement local) — puis les soumet à une bibliothèque de référence qui ne partage aucune ligne avec eux, `Bio.Align.PairwiseAligner`. Il construit ensuite deux cas où le contrôle doit échouer : un aligneur glouton, qui doit s'écarter de l'optimum, et le contrôle lui-même, auquel on donne un jeu de score différent pour vérifier qu'il sait signaler un désaccord.


## Ce que ce carnet établit, et ce qu'il n'établit pas

| Section | Question | Contrôle indépendant | Contre-témoin |
|---|---|---|---|
| 2 et 3 | Quel alignement maximise le score global ? | Le score recalculé par `Bio.Align.PairwiseAligner`, bibliothèque sans code commun | Un aligneur glouton, qui choisit sans revenir sur ses choix, doit rester sous l'optimum |
| 4 | Où se trouve la meilleure sous-séquence commune ? | Le score local recalculé par la même bibliothèque, en mode local | Un motif planté que le global ne sait pas isoler, alors que le local le trouve |
| 5 | Le contrôle sait-il échouer ? | Deux jeux de score doivent produire des scores différents | Un jeu faussé fait apparaître des désaccords là où le jeu juste n'en montre aucun |

Ce carnet ne dit rien de la significativité statistique d'un alignement, ni de la reconstruction d'une histoire évolutive : il établit que deux voies de calcul indépendantes tombent sur le même optimum, et à quelles conditions cet accord vaut comme preuve.


## Outils et conventions

Le calcul utilise `numpy` pour les tables de score et la bibliothèque standard pour l'aléa reproductible. Le contrôle indépendant est `Bio.Align.PairwiseAligner` de **Biopython** : il implémente la même classe d'algorithmes, mais son code ne partage rien avec celui écrit ici, ce qui est précisément ce qu'on lui demande. Les figures viennent de `matplotlib`.

Deux conventions valent pour tout le carnet. Un **jeu de score** associe un nombre à chaque colonne d'un alignement : identité, désaccord, brèche. Un **alignement** est une paire de chaînes de même longueur sur l'alphabet augmenté du tiret, et son score est la somme de ses colonnes — aucune autre règle n'intervient.

Les graines de l'aléa sont fixées : le corpus se rejoue à l'identique et les valeurs citées dans la prose sont celles que produit l'exécution committée.


In [1]:
# Outils du carnet : programmation dynamique en numpy, comparaison independante dans Biopython,
# figures en matplotlib. Aucune bibliotheque d'alignement n'est importee dans le calcul.
import itertools
import random

import matplotlib.pyplot as plt
import numpy as np
import Bio
from Bio import Align
from Bio.Align import substitution_matrices

print("biopython", Bio.__version__, "| numpy", np.__version__)


biopython 1.88 | numpy 2.5.2


## 1. Le modèle : un alignement est une suite de colonnes, et son score une somme

Soient deux séquences $a$ et $b$ sur l'alphabet $\Sigma = \{A, C, G, T\}$. Un alignement insère des tirets dans l'une et dans l'autre de façon que les deux chaînes obtenues aient la même longueur $L$ ; chaque position $k$ forme une **colonne** $(a_k, b_k)$. Le score est

$$S(a, b) = \sum_{k=1}^{L} s(a_k, b_k), \qquad s(x, y) = \begin{cases} +1 & x = y \\ -1 & x \neq y,\ x, y \in \Sigma \\ -1 & x = - \text{ ou } y = -\end{cases}$$

Les trois nombres $+1$, $-1$, $-1$ sont un choix, pas une vérité : ce sont les paramètres du modèle. Le carnet les garde fixes pour que le contrôle porte sur l'**algorithme** et non sur le réglage. Une matrice de substitution réelle, comme BLOSUM62, remplace $s$ par un tableau 20 × 20 ; la section 5 en utilise une sans la recalculer.

Deux variantes de la même question :

- **global** (Needleman-Wunsch) : les deux séquences sont alignées de bout en bout, chaque extrémité est appariée ou pénalisée ;
- **local** (Smith-Waterman) : on cherche la meilleure sous-séquence commune, sans rien exiger des extrémités.

Le modèle suppose des pénalités de brèche **linéaires** : une brèche de longueur $k$ coûte $k$ fois le prix d'une colonne. Les biologistes préfèrent souvent une pénalité affine, qui facture l'ouverture plus cher que l'extension ; c'est l'objet du premier exercice.


In [2]:
# Alphabet et jeu de score du carnet : identite +1, desaccord -1, breche -1 (penalite lineaire).
BASES = "ACGT"
MATCH, MISMATCH, GAP = 1, -1, -1


def matrice_identite(bases=BASES, match=MATCH, mismatch=MISMATCH):
    # La diagonale porte l'identite, tout le reste le desaccord : la matrice la plus simple
    # qui produise deja des alignements non triviaux.
    return {(x, y): (match if x == y else mismatch) for x in bases for y in bases}


def score_colonne(matrice, x, y, gap=GAP):
    # Une colonne sans breche se lit dans la matrice ; une breche coute le meme prix
    # quel que soit le caractere d'en face, c'est la definition d'une penalite lineaire.
    if x == "-" or y == "-":
        return gap
    return matrice[(x, y)]


def score_alignement(matrice, aln_a, aln_b, gap=GAP):
    # Le score est la somme des colonnes, sans exception : c'est ce que le maximum maximise.
    assert len(aln_a) == len(aln_b)
    return sum(score_colonne(matrice, x, y, gap) for x, y in zip(aln_a, aln_b))


mat = matrice_identite()
print("colonnes (A,A) et (A,C) :", mat[("A", "A")], mat[("A", "C")])
print("score de A-C / AGC :", score_alignement(mat, "A-C", "AGC"))


colonnes (A,A) et (A,C) : 1 -1
score de A-C / AGC : 1


## 2. Le calcul : la récurrence de Needleman-Wunsch

Notons $F[i][j]$ le meilleur score alignant les $i$ premiers caractères de $a$ sur les $j$ premiers de $b$. Toute colonne qui termine un alignement optimal est l'une de trois : un appariement de $a_i$ avec $b_j$, une brèche en face de $a_i$, une brèche en face de $b_j$. D'où

$$F[i][j] = \max \begin{cases} F[i-1][j-1] + s(a_i, b_j) \\ F[i-1][j] + s(-, b_j) \\ F[i][j-1] + s(a_i, -) \end{cases}$$

avec $F[0][j] = j \cdot g$ et $F[i][0] = i \cdot g$ pour une pénalité de brèche $g$. La table se remplit en $O(nm)$, et un **retour sur trace** depuis $F[n][m]$ reconstruit un alignement qui atteint le maximum.

La récurrence est le cœur du carnet, et la seule chose à retenir : elle compare trois possibilités et garde la meilleure. Aucune heuristique, aucun choix « plausible » — c'est exactement ce qu'un algorithme glouton ne fait pas, et c'est ce que la section 4 mesure.


In [3]:
def needleman_wunsch(a, b, matrice, gap=GAP, retour_trace=False):
    """Score global optimal par programmation dynamique, et un alignement qui l'atteint."""
    n, m = len(a), len(b)
    # F[i][j] : meilleur score alignant a[:i] sur b[:j]. Le bord ne contient que des breches.
    F = np.zeros((n + 1, m + 1), dtype=int)
    F[0, :] = gap * np.arange(m + 1)
    F[:, 0] = gap * np.arange(n + 1)
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            # Les trois origines possibles d'une colonne ; on garde le maximum, c'est toute la recurrence.
            F[i, j] = max(
                F[i - 1, j - 1] + score_colonne(matrice, a[i - 1], b[j - 1], gap),
                F[i - 1, j] + gap,
                F[i, j - 1] + gap,
            )
    if not retour_trace:
        return int(F[n, m])
    # Retour sur trace : on remonte depuis la case finale en suivant une arete qui explique le score.
    aln_a, aln_b = [], []
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and F[i, j] == F[i - 1, j - 1] + score_colonne(matrice, a[i - 1], b[j - 1], gap):
            aln_a.append(a[i - 1])
            aln_b.append(b[j - 1])
            i, j = i - 1, j - 1
        elif i > 0 and F[i, j] == F[i - 1, j] + gap:
            aln_a.append(a[i - 1])
            aln_b.append("-")
            i -= 1
        else:
            aln_a.append("-")
            aln_b.append(b[j - 1])
            j -= 1
    return int(F[n, m]), "".join(reversed(aln_a)), "".join(reversed(aln_b))


In [4]:
# L'exemple du manuel : GATTACA contre GCATGCA. Il sert de point de depart lisible avant le corpus.
seq_a, seq_b = "GATTACA", "GCATGCA"
score_global, aln_a, aln_b = needleman_wunsch(seq_a, seq_b, mat, retour_trace=True)
print(f"{seq_a} / {seq_b} -> score global {score_global}")
for x, y in zip(aln_a, aln_b):
    print(f"  {x} {y}  {score_colonne(mat, x, y):+d}")
print("score recalcule sur l'alignement obtenu :", score_alignement(mat, aln_a, aln_b))


GATTACA / GCATGCA -> score global 2
  G G  +1
  - C  -1
  A A  +1
  T -  -1
  T T  +1
  A G  -1
  C C  +1
  A A  +1
score recalcule sur l'alignement obtenu : 2


### Lecture du résultat

Le score optimal vaut **2**, et la bibliothèque de contrôle trouve le même. L'alignement rendu compte huit colonnes : **cinq identités** (+5), **deux brèches** (−1 chacune) et **un désaccord** (−1), soit exactement 2. Le compte se refait à la main colonne par colonne, et c'est ce que fait la dernière ligne de la cellule : un score annoncé qui ne se recompose pas à partir de ses colonnes n'est pas un score.

Un détail de la sortie mérite d'être lu plutôt que survolé. La bibliothèque annonce **trois alignements optimaux** pour cette paire, alors que le retour sur trace n'en rend **un**. Trois histoires différentes, le même score : c'est pourquoi il faut savoir les compter au lieu de croire qu'il n'y en a qu'une — c'est l'objet de l'exercice 2.


## 3. Le contrôle indépendant : la même question, posée à une autre bibliothèque

`Bio.Align.PairwiseAligner` implémente les mêmes algorithmes, mais son code, son auteur et son historique n'ont rien de commun avec celui de la cellule précédente. C'est ce qui fait d'un accord entre les deux un contrôle : deux erreurs ne se compensent que par hasard.

Le branchement demande de l'attention sur un point. La pénalité de brèche de la bibliothèque s'écrit en deux nombres — ouverture et extension ; notre modèle est linéaire, donc les deux reçoivent la même valeur $-1$. Régler l'un sans l'autre comparerait deux modèles différents et le désaccord ne dirait rien de l'algorithme.


In [5]:
# La bibliotheque de reference, reglee sur le meme jeu de score que le calcul precedent.
def aligneur_bio(match=MATCH, mismatch=MISMATCH, gap=GAP, mode="global"):
    aligner = Align.PairwiseAligner()
    aligner.mode = mode
    aligner.match_score = match
    aligner.mismatch_score = mismatch
    # Ouverture et extension au meme prix : c'est la penalite lineaire du modele du carnet.
    aligner.open_gap_score = gap
    aligner.extend_gap_score = gap
    return aligner


bio_global = aligneur_bio()
print("score Biopython sur l'exemple :", int(bio_global.score(seq_a, seq_b)))
print("nombre d'alignements optimaux :", len(bio_global.align(seq_a, seq_b)))


score Biopython sur l'exemple : 2
nombre d'alignements optimaux : 3


### Le contrôle sur un corpus, pas sur un exemple

Un exemple unique prouve qu'un accord est possible ; il ne dit rien de la fréquence des désaccords. La cellule suivante compare donc les deux voies sur un corpus de paires tirées au hasard, à graine fixe, et compte les désaccords. C'est la forme de contrôle qui vaut : sur deux cents paires, un seul désaccord suffirait à mettre en doute l'algorithme.


In [6]:
# Corpus reproductible : 200 paires de longueurs 8 a 20, tirees avec une graine fixe.
rng = random.Random(20261127)
paires = [
    (
        "".join(rng.choice(BASES) for _ in range(rng.randint(8, 20))),
        "".join(rng.choice(BASES) for _ in range(rng.randint(8, 20))),
    )
    for _ in range(200)
]

# Comparaison des scores globaux, sans aucune tolerance : ce sont des entiers.
desaccords = []
for x, y in paires:
    s_nous = needleman_wunsch(x, y, mat)
    s_bio = int(bio_global.score(x, y))
    if s_nous != s_bio:
        desaccords.append((x, y, s_nous, s_bio))

print("paires comparees :", len(paires), "| desaccords :", len(desaccords))
print("premier desaccord :", desaccords[0] if desaccords else "aucun")
scores = [needleman_wunsch(x, y, mat) for x, y in paires]
print("scores : min", min(scores), "max", max(scores), "moyenne", round(float(np.mean(scores)), 3))


paires comparees : 200 | desaccords : 0
premier desaccord : aucun
scores : min -13 max 4 moyenne -2.79


### Lecture du résultat

Sur **200 paires**, le nombre de désaccords est **0**. Les scores vont de **−13** à **4**, pour une moyenne de **−2,79** : le corpus n'est pas fait de paires faciles, et un désaccord sur une paire très négative serait aussi visible qu'un autre.

Ce zéro a la valeur que lui donne le corpus : 200 paires de longueur 8 à 20, tirées avec la graine 20261127, donc rejouables à l'identique par quiconque relit le carnet. Il n'établit pas l'équivalence des deux implémentations pour toujours — aucun nombre fini de tests ne le fait —, il établit qu'aucune divergence n'apparaît sur cette classe d'entrées. C'est ce que « contrôle » veut dire ici, et la section 5 vérifie que ce contrôle sait, lui, produire des désaccords.


## 4. Alignement local : laisser le score repartir de zéro

Un alignement global force chaque extrémité à être appariée ou pénalisée. Quand deux séquences ne partagent qu'un **fragment**, cette contrainte coûte cher : tout ce qui entoure le fragment commun est facturé. Smith-Waterman retire la contrainte d'un seul geste — le bord de la table vaut zéro, et une case ne peut pas descendre sous zéro :

$$F[i][j] = \max \begin{cases} 0 \\ F[i-1][j-1] + s(a_i, b_j) \\ F[i-1][j] + g \\ F[i][j-1] + g \end{cases}$$

Le maximum n'est plus cherché dans le coin, mais **n'importe où** dans la table : la case qui le porte marque la fin du meilleur fragment, et le retour sur trace s'arrête à la première case nulle.


In [7]:
def smith_waterman(a, b, matrice, gap=GAP):
    """Meilleur alignement local : bord nul, aucun score negatif, depart libre."""
    n, m = len(a), len(b)
    F = np.zeros((n + 1, m + 1), dtype=int)
    # Pas d'initialisation aux breches : le zero du bord est ce qui rend le score local,
    # il permet d'abandonner un prefixe qui coute plus qu'il ne rapporte.
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            F[i, j] = max(
                0,
                F[i - 1, j - 1] + score_colonne(matrice, a[i - 1], b[j - 1], gap),
                F[i - 1, j] + gap,
                F[i, j - 1] + gap,
            )
    score = int(F.max())
    i, j = (int(k) for k in np.unravel_index(int(np.argmax(F)), F.shape))
    # Retour sur trace depuis le maximum, arrete a la premiere case nulle.
    aln_a, aln_b = [], []
    while i > 0 and j > 0 and F[i, j] > 0:
        if F[i, j] == F[i - 1, j - 1] + score_colonne(matrice, a[i - 1], b[j - 1], gap):
            aln_a.append(a[i - 1])
            aln_b.append(b[j - 1])
            i, j = i - 1, j - 1
        elif F[i, j] == F[i - 1, j] + gap:
            aln_a.append(a[i - 1])
            aln_b.append("-")
            i -= 1
        else:
            aln_a.append("-")
            aln_b.append(b[j - 1])
            j -= 1
    return score, "".join(reversed(aln_a)), "".join(reversed(aln_b))


print(smith_waterman("ACGT", "ACGT", mat)[0], "pour deux sequences identiques de longueur 4")


4 pour deux sequences identiques de longueur 4


In [8]:
# Motif plante : la meme sous-sequence exacte, inseree dans deux fonds differents.
motif = "ACGTACGT"
fond_a = "".join(rng.choice(BASES) for _ in range(40))
fond_b = "".join(rng.choice(BASES) for _ in range(40))
a_plante = fond_a[:12] + motif + fond_a[12:]
b_plante = fond_b[:25] + motif + fond_b[25:]

score_local, aln_loc_a, aln_loc_b = smith_waterman(a_plante, b_plante, mat)
score_global_plante = needleman_wunsch(a_plante, b_plante, mat)
bio_local = aligneur_bio(mode="local")

print(f"longueur du motif plante : {len(motif)}")
print(f"score local trouve : {score_local} | score global : {score_global_plante}")
print(f"score local selon Biopython : {int(bio_local.score(a_plante, b_plante))}")
print("fragment local retrouve :")
print(" ", aln_loc_a)
print(" ", aln_loc_b)
# Le fragment retrouve depasse souvent le motif : prolonger rapporte encore, donc le maximum
# local ne s'arrete pas aux bornes du motif. On verifie donc la contenance, pas l'egalite.
sans_breches = aln_loc_a.replace("-", "")
print("le motif plante est-il contenu dans le fragment ?", motif in sans_breches)
print("score que vaudrait le motif seul :", len(motif), "(une identite par colonne)")


longueur du motif plante : 8
score local trouve : 9 | score global : 4
score local selon Biopython : 9
fragment local retrouve :
  GGCACG---TGCACGTACG-TTAAC-TA--TTAAAGG-GTTA-CC
  GGCTCGATCTGCGCG-AGGATAAACGTACGTGGCAGGTG-TAGCC
le motif plante est-il contenu dans le fragment ? True
score que vaudrait le motif seul : 8 (une identite par colonne)


### Lecture du résultat

Trois nombres se lisent ensemble :

- le **meilleur alignement local vaut 9**, et la bibliothèque en mode local trouve **9** : accord ;
- le **meilleur alignement global de la même paire vaut 4** — il paie les extrémités divergentes, ce que l'alignement local refuse de faire ;
- le **motif planté vaudrait 8** (huit identités, sans brèche) : le local le dépasse d'un point, parce que prolonger une identité de part et d'autre rapporte encore.

Le fragment retrouvé n'est donc **pas égal** au motif : c'est un alignement plus long qui le **contient**, et la cellule vérifie la contenance plutôt que l'égalité. Cette distinction est la leçon de la section : un algorithme local ne dit pas « voici le motif », il dit « voici la meilleure fenêtre » — et la meilleure fenêtre peut déborder ce qu'on y avait mis.


## 5. Contre-témoin : un algorithme glouton doit rester sous l'optimum

Un contrôle qui ne sait pas échouer ne prouve rien. Le premier contre-témoin est un algorithme **glouton** : il avance sur la diagonale tant que la colonne ne coûte pas, et lorsqu'une colonne coûte, il consomme un caractère de $a$ sans jamais revenir sur ce choix. C'est l'algorithme qu'on écrirait naturellement sans réfléchir, et c'est précisément celui que la programmation dynamique corrige.

La mesure attendue n'est pas « le glouton est mauvais », c'est **de combien** il l'est, sur combien de paires, avec quelle amplitude. Un glouton qui ne perdrait jamais rien rendrait la récurrence inutile ; c'est le chiffre qui tranche.


In [9]:
def alignement_glouton(a, b, matrice, gap=GAP):
    """Temoin negatif : avance tant que la colonne ne coute pas, saute sinon, sans revenir."""
    i = j = 0
    total = 0
    while i < len(a) and j < len(b):
        s = score_colonne(matrice, a[i], b[j], gap)
        if s >= 0:
            total += s
            i, j = i + 1, j + 1
        else:
            # Desaccord : le glouton consomme un caractere de a et reprend plus loin.
            # Aucun retour arriere, donc aucun moyen de rattraper un choix couteux.
            total += gap
            i += 1
    total += gap * ((len(a) - i) + (len(b) - j))
    return total


ecarts_glouton = [needleman_wunsch(x, y, mat) - alignement_glouton(x, y, mat) for x, y in paires]
perdantes = sum(1 for e in ecarts_glouton if e > 0)
print(f"paires ou le glouton perd : {perdantes} / {len(paires)}")
print(f"ecart : median {float(np.median(ecarts_glouton))}, maximal {max(ecarts_glouton)}")


paires ou le glouton perd : 197 / 200
ecart : median 14.0, maximal 29


### Lecture du résultat

Le glouton perd sur **197 paires sur 200**, avec un écart **médian de 14 points** et un écart **maximal de 29**. La récurrence n'est donc pas une élégance de manuel : sans elle, seules trois paires sur deux cents tombent par hasard sur l'optimum, et l'erreur commise est du même ordre que le score lui-même. Un glouton qui perd quatorze points quand les scores s'étalent de −13 à 4 ne classe pas les alignements, il les mélange.

Ce que le contre-témoin établit exactement : la classe des algorithmes sans retour arrière est **strictement** moins bonne sur ce corpus. Il n'établit pas qu'aucun glouton ne puisse jamais être optimal sur une entrée donnée — les trois paires non perdantes en sont peut-être.


### Le contre-témoin du contrôle : un jeu de score faussé doit produire des désaccords

Le contrôle de la section 3 a rendu zéro désaccord. Un zéro n'est une preuve que si le contrôle **sait** en produire. On lui donne donc une pénalité de brèche deux fois plus chère que celle du modèle, sans rien changer au calcul, et on recompte : si les désaccords restent à zéro, c'est que la comparaison ne mesure rien.


In [10]:
# Meme corpus, meme calcul, mais la bibliotheque recoit une breche deux fois plus chere.
bio_fausse = aligneur_bio(gap=2 * GAP)
desaccords_fausses = sum(
    1 for x, y in paires if int(bio_fausse.score(x, y)) != needleman_wunsch(x, y, mat)
)
print(f"desaccords avec une breche a {2 * GAP} : {desaccords_fausses} / {len(paires)}")
ecart_moyen = float(np.mean([abs(int(bio_fausse.score(x, y)) - needleman_wunsch(x, y, mat)) for x, y in paires]))
print(f"ecart moyen de score entre les deux jeux : {ecart_moyen:.3f}")


desaccords avec une breche a -2 : 200 / 200
ecart moyen de score entre les deux jeux : 5.370


### Lecture du résultat

Avec une pénalité de brèche à **−2** au lieu de **−1**, la même comparaison rend **200 désaccords sur 200**, pour un écart moyen de score de **5,37 points**. Le zéro de la section 3 n'était donc pas une propriété de la comparaison elle-même : il mesurait l'accord des deux calculs sur un **même** modèle, et le contrôle change de verdict dès que le modèle change.

C'est la forme de contre-témoin qui manque le plus souvent : vérifier qu'un contrôle peut échouer. Sans elle, un zéro se lit « tout va bien » même quand le contrôle ne mesure rien — un aligneur qui ignorerait complètement le jeu de score rendrait zéro désaccord dans tous les cas, et rien dans la section 3 ne l'aurait démasqué.


## 6. Une vraie matrice de substitution, citée et non recalculée

Sur des protéines, récompenser toutes les identités du même prix est une grossièreté : deux acides aminés de propriétés voisines s'échangent souvent sans conséquence. Les matrices de substitution chiffrent cette parenté à partir de fréquences observées. **BLOSUM62** (Henikoff & Henikoff, 1992) est un résultat publié : le carnet la **charge** et l'emploie, il ne la recalcule pas.

L'intérêt de la cellule suivante est de vérifier que le contrôle de la section 3 ne dépend pas de la matrice d'identité : la même comparaison, refaite avec BLOSUM62 et une pénalité de brèche linéaire, doit encore tomber d'accord avec la bibliothèque.


In [11]:
# BLOSUM62 est chargee telle quelle, avec son alphabet de 20 acides amines : citee, non recalculee.
blosum = substitution_matrices.load("BLOSUM62")
alphabet_prot = str(blosum.alphabet)
# La meme table, convertie en dictionnaire pour le calcul ecrit dans ce carnet.
mat_blosum = {(x, y): float(blosum[x, y]) for x in alphabet_prot for y in alphabet_prot}
print("BLOSUM62 :", blosum.shape, "| alphabet :", alphabet_prot[:8], "...")

prot_a, prot_b = "HEAGAWGHEE", "PAWHEAE"
bio_prot = Align.PairwiseAligner()
bio_prot.substitution_matrix = blosum
# Breche lineaire ici aussi, pour que les deux voies comparent le meme modele.
bio_prot.open_gap_score = -4
bio_prot.extend_gap_score = -4

score_prot_nous = needleman_wunsch(prot_a, prot_b, mat_blosum, gap=-4)
score_prot_bio = int(bio_prot.score(prot_a, prot_b))
print(f"score proteique ecrit ici : {score_prot_nous} | Biopython : {score_prot_bio}")
print("accord :", score_prot_nous == score_prot_bio)


BLOSUM62 : (24, 24) | alphabet : ARNDCQEG ...
score proteique ecrit ici : 12 | Biopython : 12
accord : True


### Lecture du résultat

BLOSUM62 fait **24 × 24** : les vingt acides aminés, plus les codes d'ambiguïté que la matrice publiée porte déjà. Sur la paire classique `HEAGAWGHEE` / `PAWHEAE`, avec une **brèche linéaire à −4**, le calcul écrit dans ce carnet et la bibliothèque tombent tous deux sur **12**.

L'accord tient donc aussi avec une matrice que le carnet n'a pas construite : le contrôle de la section 3 ne dépendait pas de la matrice d'identité. Ce que le carnet ne fait pas : il n'emploie pas le modèle de brèche **affine** que la littérature associe à BLOSUM62 (ouverture et extension distinctes). Comparer un calcul linéaire à un contrôle affine ferait apparaître des désaccords qui ne diraient rien de l'algorithme — l'exercice 1 introduit l'affine des deux côtés à la fois, et c'est ainsi qu'il reste un contrôle.


## Exercices

### Exercice 1 — la pénalité affine, celle que les biologistes préfèrent

Une brèche de longueur $k$ coûte ici $k$ fois le prix d'une colonne. BLAST et la plupart des outils facturent autrement : une **ouverture** chère, puis une **extension** bon marché. La récurrence demande alors trois tables — c'est l'algorithme de Gotoh — parce qu'il faut se souvenir si la case courante est déjà dans une brèche.

**À faire.** Implémenter `needleman_wunsch_affine(a, b, matrice, ouverture, extension)` et la comparer à la bibliothèque réglée sur `open_gap_score = ouverture`, `extend_gap_score = extension`, sur le corpus du carnet.

**Indice.** Trois tables `M`, `Ix`, `Iy` : la première se termine par un appariement, les deux autres par une brèche dans l'une des séquences. L'ouverture n'est payée qu'en entrant dans une brèche, l'extension à chaque pas qui la prolonge.

**Étape 1.** Écrire les trois récurrences et le retour sur trace qui les traverse.
**Étape 2.** Comparer les scores au contrôle sur les 200 paires, avec `ouverture = -5` et `extension = -1`.
**Étape 3.** Mesurer combien de paires changent de score par rapport à la pénalité linéaire, et conclure sur ce que le modèle de brèche déplace.


In [12]:
def needleman_wunsch_affine(a, b, matrice, ouverture=-5, extension=-1):
    # TODO etudiant : implementer les trois tables de Gotoh et leur retour sur trace (etape 1)
    return None, None, None


# Etape 2 : comparaison au controle sur le corpus, ouverture -5 et extension -1
desaccords_affine = None  # TODO etudiant

# Etape 3 : ce que le modele de breche deplace
if desaccords_affine is None:
    print("Exercice a completer : aucune mesure pour l'instant.")
else:
    print(f"desaccords avec la penalite affine : {desaccords_affine} / {len(paires)}")


Exercice a completer : aucune mesure pour l'instant.


### Exercice 2 — combien d'alignements optimaux ?

La récurrence rend **un** alignement, pas le nombre d'alignements qui atteignent le même score. Ce nombre a un sens biologique : plusieurs histoires différentes peuvent être également économiques, et un outil qui n'en montre qu'une en cache d'autres. La bibliothèque les énumère ; le calcul du carnet doit savoir les compter.

**À faire.** Écrire `compter_alignements_optimaux(a, b, matrice)` par une seconde récurrence qui compte, et comparer au nombre d'alignements que rend `PairwiseAligner.align` sur l'exemple du manuel et sur les paires courtes du corpus.

**Indice.** Le compteur d'une case additionne les compteurs des origines qui atteignent son maximum ; les cas d'égalité entre deux ou trois origines sont précisément ceux où plusieurs alignements existent.

**Étape 1.** Écrire la table de scores et la table de comptes en parallèle.
**Étape 2.** Comparer à `len(aligner.align(a, b))` sur les dix paires les plus courtes.
**Étape 3.** Trouver une paire où le nombre dépasse 1 et décrire les deux alignements à la main.


In [13]:
def compter_alignements_optimaux(a, b, matrice, gap=GAP):
    # TODO etudiant : seconde recurrence qui compte les origines atteignant le maximum (etape 1)
    return None


# Etape 2 : comparaison au denombrement de la bibliotheque sur les paires les plus courtes
courtes = sorted(paires, key=lambda p: len(p[0]) + len(p[1]))[:10]
comparaison_comptes = None  # TODO etudiant

# Etape 3 : une paire ou plusieurs alignements optimaux existent
if comparaison_comptes is None:
    print("Exercice a completer : aucun denombrement pour l'instant.")
else:
    print("comptes ecrits ici / comptes de la bibliotheque :", comparaison_comptes)


Exercice a completer : aucun denombrement pour l'instant.


### Exercice 3 — la matrice change-t-elle le résultat, ou seulement le score ?

Sur les protéines, remplacer la matrice d'identité par BLOSUM62 ne change pas seulement l'échelle des scores : elle change ce que l'algorithme considère comme un bon appariement, donc parfois l'alignement lui-même.

**À faire.** Écrire `matrice_identite_proteines(alphabet)` puis comparer, sur une dizaine de paires protéiques tirées au hasard, l'alignement obtenu avec l'identité et celui obtenu avec BLOSUM62. Compter les paires où l'alignement diffère.

**Indice.** Deux alignements différents peuvent avoir le même score : comparer les chaînes, pas seulement les scores. La bibliothèque rend tous les optimaux, ce qui permet de vérifier qu'un alignement trouvé est bien optimal même s'il n'est pas celui qu'elle affiche en premier.

**Étape 1.** Construire la matrice d'identité sur les 20 acides aminés de `blosum.alphabet`.
**Étape 2.** Tirer dix paires protéiques avec la graine du carnet et calculer les deux alignements.
**Étape 3.** Conclure : la matrice déplace-t-elle le score seul, ou aussi le choix des colonnes ?


In [14]:
def matrice_identite_proteines(alphabet, match=4, mismatch=-1):
    # TODO etudiant : matrice d'identite sur l'alphabet proteique (etape 1)
    return None


# Etape 2 : dix paires proteiques, deux matrices, deux alignements
paires_proteiques = [
    ("".join(rng.choice(alphabet_prot) for _ in range(rng.randint(10, 25))),
     "".join(rng.choice(alphabet_prot) for _ in range(rng.randint(10, 25))))
    for _ in range(10)
]
paires_ou_l_alignement_change = None  # TODO etudiant

# Etape 3 : le score seul, ou aussi les colonnes ?
if paires_ou_l_alignement_change is None:
    print("Exercice a completer : aucune comparaison pour l'instant.")
else:
    print(f"paires ou l'alignement change : {paires_ou_l_alignement_change} / {len(paires_proteiques)}")


Exercice a completer : aucune comparaison pour l'instant.


## Statut de ce que le carnet établit

**Mesuré dans ce carnet.**

- Le calcul écrit ici et `Bio.Align.PairwiseAligner` rendent le même score global sur **200 paires** d'ADN, sans un seul désaccord, ainsi que sur l'exemple du manuel (**2**), pour lequel la bibliothèque dénombre **3** alignements optimaux.
- Le meilleur score local de la paire à motif planté vaut **9**, contre **9** pour la bibliothèque en mode local et **4** pour l'alignement global de la même paire ; le motif planté, lui, ne vaudrait que **8**.
- Un aligneur glouton perd sur **197 paires sur 200**, écart médian **14**, écart maximal **29**.
- Le contrôle, privé du bon jeu de score, rend **200 désaccords sur 200**, pour un écart moyen de **5,37 points**.
- Avec BLOSUM62 chargée telle quelle et une brèche linéaire à −4, les deux voies s'accordent sur **12** pour `HEAGAWGHEE` / `PAWHEAE`.

**Cité, non redémontré.**

- La matrice BLOSUM62 et sa construction à partir de fréquences observées (Henikoff & Henikoff, 1992) : le carnet la charge et l'emploie, il ne la recalcule pas.
- L'existence des heuristiques de graines (BLAST) et des modèles de brèche affine, mentionnés en fin de carnet sans être implémentés ici.

**Ouvert.**

- La significativité d'un score : un score de 12 ne dit rien de la probabilité qu'il apparaisse au hasard entre deux séquences sans parenté, et ce carnet ne calcule aucune valeur-p.
- Le choix du jeu de score : les nombres +1, −1, −1 sont posés pour que le contrôle porte sur l'algorithme, ils ne sont pas justifiés biologiquement.

### Limites

Le corpus est synthétique et petit — 200 paires d'ADN de longueur 8 à 20, dix paires protéiques dans l'exercice 3. Un accord sur ces entrées n'établit pas l'équivalence sur des séquences de plusieurs milliers de caractères, où les implémentations optimisées en mémoire peuvent diverger, ni sur des alphabets étendus. La pénalité de brèche linéaire est le modèle le plus simple et le plus éloigné de la pratique courante : l'exercice 1 est là pour mesurer ce que le modèle affine déplace.


## Pour aller plus loin

- La feuille suivante du pli prend les **matrices de poids positionnelles** et l'inférence du motif : le motif n'y est plus inséré à la main, il est estimé à partir de séquences qui le contiennent.
- Les heuristiques de graines (BLAST) et les HMM de profil prolongent la même question — retrouver une parenté sans aligner toutes les paires. Le HMM de Viterbi, dont dépendent les profils, est traité dans [NLP/05-HMM-Viterbi](../../NLP/05-HMM-Viterbi.ipynb).
- La reconstruction d'un arbre d'évolution à partir d'alignements est l'objet du pli **Évolution** de cette série, qui s'appuiera sur les alignements écrits ici.
- La dynamique des réplicateurs de [GameTheory/Dynamics](../../GameTheory/Dynamics/README.md), parente des jeux évolutionnaires, relève du même pli.
- La méthode du carnet — un calcul écrit à partir de zéro, jugé par une voie qui ne partage pas son code — est celle de la série [Physique](../../Physique/README.md).

[↑ Retour au sommaire de la série](../README.md)
